# Conformal Triage — Phases 1+2 in Colab, **v2**

Changes vs v1: PAD-UFES-20 is now downloaded from the **ISIC Archive (collection 406, official mirror)**
with `isic-cli` — the Mendeley link died. Everything is resumable: what is already downloaded/extracted is skipped.

**Before running:** `Runtime → Change runtime type → T4 GPU`. Then `Run all`.
Takes ~35–50 min (the ISIC 2019 extraction goes quiet for stretches: that is normal, do not interrupt it).
At the end ~61 MB are left in `conformal-triage/emb/` of your Google Drive: the only input notebook 02 needs.


In [ ]:
# 1) GPU + dependencies
!nvidia-smi -L
!pip -q install timm==0.9.16 open_clip_torch gdown isic-cli
!mkdir -p /content/data/isic2019 /content/data/hiba /content/data/pad /content/emb /content/ckpt


In [ ]:
%%bash
# 2) ISIC 2019 (9.1 GB, official challenge links) — skipped if already there
cd /content/data/isic2019
N=$(find . -name '*.jpg' 2>/dev/null | wc -l)
if [ "$N" -ge 25331 ]; then echo "ISIC 2019 already there ($N jpg), skipping"; else
  wget -qc https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip
  wget -qc https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_GroundTruth.csv
  unzip -qn ISIC_2019_Training_Input.zip
  echo "ISIC 2019: $(find . -name '*.jpg' | wc -l) jpg (expected 25331)"
fi


In [ ]:
# 3) HIBA and PAD-UFES-20, both from the ISIC Archive with isic-cli.
#    Each collection ID is looked up by name (PAD is the official mirror of the Mendeley dataset).
import subprocess, re, glob, os

def isic_collection_id(needle):
    out = subprocess.run(['isic','collection','list'], capture_output=True, text=True).stdout
    line = next(l for l in out.splitlines() if needle.lower() in l.lower())
    nums = re.findall(r'\d+', line.split('│')[1] if '│' in line else line)
    return nums[0], line.strip()[:100]

def fetch(needle, dest, expected):
    imgs = glob.glob(f'{dest}/images/**/*.*', recursive=True)
    if len(imgs) >= expected:
        print(f'{dest}: {len(imgs)} files already there, skipping'); return
    cid, line = isic_collection_id(needle)
    print(f'collection "{needle}" -> id {cid} | {line}')
    os.makedirs(f'{dest}/images', exist_ok=True)
    subprocess.run(['isic','metadata','download','--collections',cid], cwd=dest, check=False)
    subprocess.run(['isic','image','download','--collections',cid,f'{dest}/images/'], check=True)
    print(f'{dest}: {len(glob.glob(dest+chr(47)+"images"+chr(47)+"**"+chr(47)+"*.*", recursive=True))} files (expected ~{expected})')

fetch('hospital italiano', '/content/data/hiba', 1616)
fetch('pad-ufes', '/content/data/pad', 2298)
# rename the metadata csv files left by isic-cli, to take them to the Drive
for src, name in [('/content/data/hiba','hiba_isic_metadata.csv'), ('/content/data/pad','pad_isic_metadata.csv')]:
    for f in glob.glob(f'{src}/*.csv'):
        os.replace(f, f'/content/emb/{name}'); print('metadata ->', name)


In [ ]:
%%bash
# 4) PanDerm: repo + ViT-L checkpoint (Google Drive from the official README) — skipped if already there
cd /content
[ -d PanDerm ] || git clone -q https://github.com/SiyuanYan1/PanDerm
CKPT=/content/ckpt/panderm_ll_data6_checkpoint-499.pth
if [ -s "$CKPT" ]; then echo 'checkpoint already there, skipping'; else
  gdown 1SwEzaOlFV_gBKf2UzeowMC8z9UH7AQbE -O "$CKPT" || \
    echo 'If gdown failed on quota: open https://drive.google.com/file/d/1SwEzaOlFV_gBKf2UzeowMC8z9UH7AQbE/view , "Add shortcut to My Drive", mount your Drive and copy it to /content/ckpt/'
fi
ls -lh /content/ckpt/


In [ ]:
# 5) Extraction (frozen PanDerm, one pass per image). Skips what was already extracted.
import sys, os, time, json, contextlib, io, numpy as np, pandas as pd, torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
sys.path.insert(0, '/content/PanDerm/classification')
with contextlib.redirect_stdout(io.StringIO()):   # silences the model's huge print
    from models import get_encoder
    class A: pretrained_checkpoint = '/content/ckpt/panderm_ll_data6_checkpoint-499.pth'
    model, tfm = get_encoder(A(), model_name='PanDerm_Large_LP')
model.eval().cuda()
print('model loaded: ViT-L, 1024-d embedding')
EXTS = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}

def extract(img_dir, batch=64, workers=2):
    paths = sorted(p for p in Path(img_dir).rglob('*') if p.suffix.lower() in EXTS)
    class DS(Dataset):
        def __len__(s): return len(paths)
        def __getitem__(s, i):
            try: img = Image.open(paths[i]).convert('RGB')
            except Exception: img = Image.new('RGB', (224,224))
            return tfm(img), paths[i].stem
    dl = DataLoader(DS(), batch_size=batch, num_workers=workers, pin_memory=True)
    F, ids, t0 = [], [], time.time()
    with torch.no_grad():
        for bi,(x,st) in enumerate(dl):
            with torch.autocast('cuda', dtype=torch.float16):
                f = model.forward_features(x.cuda(non_blocking=True), is_train=False)
            F.append(f.float().cpu().numpy()); ids.extend(st)
            if bi % 10 == 0: print(f'  {min((bi+1)*batch,len(paths))}/{len(paths)} ({(bi+1)*batch/max(time.time()-t0,1e-9):.0f} img/s)', flush=True)
    return np.array(ids), np.concatenate(F).astype(np.float16)

def check(n, expected, name):
    print(('OK' if n == expected else f'WARNING: {name} has {n}, expected {expected}'), '-', name, n)

if not os.path.exists('/content/emb/hiba_emb.npz'):
    ids, F = extract('/content/data/hiba/images'); check(len(ids), 1616, 'hiba')
    np.savez_compressed('/content/emb/hiba_emb.npz', ids=ids, features=F)
else: print('hiba already extracted, skipping')
if not os.path.exists('/content/emb/pad_emb.npz'):
    ids, F = extract('/content/data/pad/images'); check(len(ids), 2298, 'pad')
    np.savez_compressed('/content/emb/pad_emb.npz', ids=ids, features=F)
else: print('pad already extracted, skipping')
if not os.path.exists('/content/emb/isic2019_emb_part0.npz'):
    gt = pd.read_csv('/content/data/isic2019/ISIC_2019_Training_GroundTruth.csv')
    CLS = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
    lab = dict(zip(gt['image'], gt[CLS].values.argmax(1)))
    ids, F = extract('/content/data/isic2019'); check(len(ids), 25331, 'isic2019')
    L = np.array([lab.get(i, -1) for i in ids], dtype=np.int16)
    assert (L >= 0).all(), 'ids without label in ISIC 2019'
    for k, sl in enumerate(np.array_split(np.arange(len(ids)), 3)):
        np.savez_compressed(f'/content/emb/isic2019_emb_part{k}.npz', ids=ids[sl], features=F[sl], labels=L[sl], classes=np.array(CLS))
else: print('isic2019 already extracted, skipping')
json.dump({'dim': 1024, 'ckpt': 'panderm_ll_data6_checkpoint-499.pth', 'dtype': 'float16',
           'pad_source': 'ISIC Archive collection 406 (official mirror of PAD-UFES-20)',
           'date': time.strftime('%Y-%m-%d %H:%M')},
          open('/content/emb/extract_meta.json','w'), indent=2)
print('extraction complete')


In [ ]:
# 6) Save to your Google Drive (~61 MB) and list
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p '/content/drive/MyDrive/conformal-triage/emb'
!cp /content/emb/* '/content/drive/MyDrive/conformal-triage/emb/'
!ls -lh '/content/drive/MyDrive/conformal-triage/emb'
print('DONE. Artefacts in Drive: conformal-triage/emb/ (input of notebook 02).')
